# Agents & Decisions — slide companion
Executable examples for the **Agents & Decisions** learning session. The slide numbers below match the current 21-slide deck. Checked against official documentation on **3 November 2026**.

**Start here:** select the project's `.venv` Python kernel, run the setup cells, enable only the demos you want, then run those cells in order. Your root `.env` supplies the API key. All live calls start disabled; skipped cells do not contain simulated API results.

| Slides | Notebook demonstration |
|---|---|
| 4–8 | Inline managed agent, saved agent, retrieval across Python instances |
| 9–12 | Single CVL, image evidence, multi-CVL |
| 16 | Predicate and an easy sentiment score |
| 17–19 | Separate Agents SDK, Azure Decisions, saved session items |
| 20 | Hosted environment costs and explicit cleanup |

For a short live showcase: setup → inline agent → single CVL → image → multi-CVL. Saved-agent and appendix examples can be explored afterwards.

## 1. Setup — slide 5
From this directory, install the existing requirements and launch Jupyter:

```powershell
.\.venv\Scripts\python -m pip install -U -r requirements.txt
.\.venv\Scripts\python -m jupyterlab
```

Use Python 3.10+ with `openai>=3.26.0,<4`. The requirements install both `openai` and the separate `openai-agents` package used by the optional SDK section. The managed Agents API needs no prior agent creation in the Platform UI.

In [ ]:
import json
import os
from base64 import b64encode
from importlib.metadata import version
from pathlib import Path
from time import perf_counter

from dotenv import load_dotenv
from IPython.display import Image, display
from openai import OpenAI

# Locate the project even if Jupyter was launched from a child directory.
ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "requirements.txt").exists()
     and (p / "outputs/learning-session").is_dir()),
    None,
)
if ROOT is None:
    raise RuntimeError("Start Jupyter from the test-openai-agents-decisions project.")

load_dotenv(ROOT / ".env", override=False)  # Existing environment variables win.
client = OpenAI(timeout=180, max_retries=0) if os.getenv("OPENAI_API_KEY") else None

DECISION_MODEL = "gpt-6-luna"
AGENT_MODEL = "gpt-6-astra"
EVIDENCE = "Navy blue jacket. Suitable for hiking and cycling."
TASK = "Write and run Python to find the average of 12, 18 and 24. Explain it."
IMAGE_PATH = ROOT / "outputs/learning-session/assets/jacket-evidence.png"

def require_client():
    if client is None:
        raise RuntimeError("Set OPENAI_API_KEY in the root .env and rerun setup.")
    return client

print("OpenAI SDK:", version("openai"))
print("API key loaded:", client is not None)  # Never display the key.
print("Image evidence found:", IMAGE_PATH.is_file())

In [ ]:
# Enable a section, rerun this cell, then run its example cells.
# Rerunning enabled cells can submit new billable requests.
RUN_INLINE_AGENT = False
RUN_SAVED_AGENT = False
RUN_RETRIEVAL = False
RUN_DECISIONS = False
RUN_AGENTS_SDK = False
RUN_AZURE_DECISIONS = False

SHOW_EVENT_JSON = True  # Full payloads, including text/tool data, as in slide 6.
# Default matches slide 6 (medium, 4 GB). For a smaller sandbox add:
# "container_size": "small"
HOSTED_ENVIRONMENT = {"type": "openai_hosted"}

# Optional: use existing managed IDs copied from the Platform or another process.
# These must be Agents API resources accessible to this API key/project.
EXISTING_AGENT_ID = None
SESSION_TO_INSPECT = None

### A little local state for the retrieval demo
The next helper stores only resource IDs in `.agents_decisions_showcase_state.json` (gitignored). Restarting the kernel does not lose them. API keys and model outputs are never written to this file. Saved agents are reused on subsequent runs; each enabled session-creation cell starts a new conversation.

In [ ]:
STATE_PATH = ROOT / ".agents_decisions_showcase_state.json"
state = json.loads(STATE_PATH.read_text(encoding="utf-8")) if STATE_PATH.exists() else {}

def remember(**ids):
    state.update(ids)
    STATE_PATH.write_text(json.dumps(state, indent=2), encoding="utf-8")

print("Previously saved ID slots:", sorted(state))

## 2. Inline managed agent — slides 4 and 6
Here the configuration lives inside the session request. The agent writes and runs Python in an OpenAI-hosted sandbox; that environment is separate from your notebook computer.

Enable `RUN_INLINE_AGENT`. Watch the **actual JSON event bodies**, then the collected answer. The expected arithmetic result is 18; use the tool events to see what actually ran. `agent_events` retains the payloads in memory for inspection.

A session ID is saved as soon as it appears so you can retrieve the session if streaming is interrupted. Closing the stream does not cancel remote work. Session idleness alone does not prove the turn succeeded; inspect its terminal event and tool output.

Source: [Run and continue sessions](https://developers.openai.com/api/docs/guides/agents-api/sessions).

In [ ]:
agent_events = []
inline_result = None

if RUN_INLINE_AGENT:
    api = require_client()
    terminal_event = None
    with api.beta.agents.sessions.create(
        agent={
            "model": AGENT_MODEL,
            "instructions": "Run code and report the actual output.",
        },
        environment=HOSTED_ENVIRONMENT,
        input=TASK,
        stream=True,
    ).with_result_collection() as events:
        for event in events:
            payload = event.model_dump(mode="json")
            agent_events.append(payload)
            sid = payload.get("session_id") or (payload.get("session") or {}).get("id")
            if sid and sid != state.get("inline_session_id"):
                remember(inline_session_id=sid)
            if event.type in {
                "agent.session.turn.completed",
                "agent.session.turn.failed",
                "agent.session.turn.cancelled",
            }:
                terminal_event = event.type
            if SHOW_EVENT_JSON:
                print(event.to_json(indent=None), flush=True)
            elif event.type == "agent.session.turn.output_text.delta":
                print(payload.get("delta", ""), end="", flush=True)
        inline_result = events.get_final_result()

    remember(inline_session_id=inline_result.session_id)
    print("\nTerminal event:", terminal_event or "No terminal turn event observed")
    print("Session:", inline_result.session_id)
    print("Collected answer:", inline_result.output_text)
else:
    print("Skipped: set RUN_INLINE_AGENT = True for the hosted demonstration.")

In [ ]:
# The full event bodies remain available after the stream ends.
if agent_events:
    print("Event count:", len(agent_events))
    print("Event types:", list(dict.fromkeys(e["type"] for e in agent_events)))
    # Inspect any event, including tool-related payloads:
    print(json.dumps(agent_events[-1], indent=2))
else:
    print("Run the inline demo to inspect its real events.")

## 3. Saved managed agent — slides 4 and 7
Create a reusable remote configuration, then start a session using its `agent_id`. The cell reuses the locally saved ID, or `EXISTING_AGENT_ID`, before creating a new agent. A compatible agent already created in the Platform can therefore be used here too.

Enable `RUN_SAVED_AGENT`. This example uses `environment={"type": "none"}`: a model explanation without a hosted code sandbox. Session creation returns asynchronously, so the answer may not be ready immediately. Retrieve its status and items in the next section.

A saved agent is configuration; a session holds one conversation. Updating saved configuration applies to new sessions. This is separate from the local `Agent()` object in the Agents SDK appendix.

Source: [Configuring Agents](https://developers.openai.com/api/docs/guides/agents-api/configuration).

New agents are named **Technical explainer** using the optional `name` parameter (up to 128 characters). The name is a human-readable label; API calls still use the returned `agent_id`. Reused agents keep their existing name. To rename one, uncomment the `agents.update(...)` line in the next cell before creating its session. See the [create](https://developers.openai.com/api/reference/python/resources/beta/subresources/agents/methods/create) and [update](https://developers.openai.com/api/reference/python/resources/beta/subresources/agents/methods/update) references.

In [ ]:
SAVED_AGENT_NAME = "Technical explainer"

if RUN_SAVED_AGENT:
    api = require_client()
    saved_agent_id = EXISTING_AGENT_ID or state.get("saved_agent_id")
    if saved_agent_id:
        saved = api.beta.agents.retrieve(saved_agent_id)
        # Optional: uncomment to rename an existing agent before creating a session.
        # saved = api.beta.agents.update(saved.id, name=SAVED_AGENT_NAME)
        print("Reusing agent:", saved.id)
    else:
        saved = api.beta.agents.create(
            name=SAVED_AGENT_NAME,
            model=AGENT_MODEL,
            instructions="Explain technical issues clearly.",
        )
        remember(saved_agent_id=saved.id)
        print("Created agent:", saved.id)

    print("Agent name:", saved.name)
    saved_session = api.beta.agents.sessions.create(
        agent_id=saved.id,
        environment={"type": "none"},
        input="Explain why a file export might fail.",
    )
    remember(saved_agent_id=saved.id, saved_session_id=saved_session.id)
    print("Session:", saved_session.id)
    print("Initial status:", saved_session.status)
else:
    print("Skipped: set RUN_SAVED_AGENT = True to create/reuse a saved agent.")

## 4. Retrieve from another Python instance — slides 8 and 19
Enable `RUN_RETRIEVAL`. This uses a **new OpenAI client** and reloads IDs from disk, just as a new process would. After restarting the kernel, run setup, controls, state, and this section; creation cells can remain disabled.

To inspect a particular session, set `SESSION_TO_INSPECT`; otherwise the saved-agent session takes priority over the inline session. If only the inline example ran, there is no reusable saved agent to retrieve.

Retrieval does not submit a new turn. If a session is still working, rerun this cell later. Only the first page of up to ten saved items is shown, keeping the demo compact.

Source: [Manage sessions](https://developers.openai.com/api/docs/guides/agents-api/sessions/manage).

In [ ]:
if RUN_RETRIEVAL:
    require_client()
    persisted = json.loads(STATE_PATH.read_text(encoding="utf-8")) if STATE_PATH.exists() else {}
    agent_id = EXISTING_AGENT_ID or persisted.get("saved_agent_id")
    session_id = (
        SESSION_TO_INSPECT
        or persisted.get("saved_session_id")
        or persisted.get("inline_session_id")
    )
    with OpenAI(timeout=180, max_retries=0) as later_client:
        if agent_id:
            retrieved_agent = later_client.beta.agents.retrieve(agent_id)
            print("Agent:", retrieved_agent.id)
            print("Name:", retrieved_agent.name)
            print("Model:", retrieved_agent.model)
            print("Instructions:", retrieved_agent.instructions)
        else:
            print("No saved agent ID; inline sessions do not create a reusable agent.")
        if session_id:
            retrieved_session = later_client.beta.agents.sessions.retrieve(session_id)
            print("Session:", retrieved_session.id, "Status:", retrieved_session.status)
            print("Saved items (first page):")
            items = later_client.beta.agents.sessions.items.list(
                session_id, order="asc", limit=10
            )
            for item in items.data:
                print(item.model_dump_json(indent=2))
        else:
            print("Create a session first or set SESSION_TO_INSPECT.")
else:
    print("Skipped: set RUN_RETRIEVAL = True to retrieve remote resources.")

## 5. Decisions: one bounded judgment — slides 9 and 10
A **choice** returns one supplied value. For a single PIM controlled vocabulary list (CVL), map supplier wording to a canonical key. Here, “navy blue” should map to `blue`.

Enable `RUN_DECISIONS` for all four Decisions examples. Each cell prints real typed answers, token usage, and client-observed elapsed time. Any question can return a refusal. `needs_review` is an application fallback, **not a valid PIM color key**. These examples produce draft dictionaries; they do not write to a PIM.

OpenAI reports **about 10× faster typed answers than the Responses API**. This is the provider's comparison; a single timed demo does not validate that ratio. Actual latency depends on the workload.

Source: [Decisions](https://developers.openai.com/api/docs/guides/decisions).

In [ ]:
def show_decision(decision, started):
    print(f"Elapsed: {perf_counter() - started:.3f}s (client-observed)")
    for answer in decision.answers:
        print(answer.model_dump_json(indent=2))
    print("Usage:", decision.usage.model_dump())

def color_draft(decision, valid_keys):
    # Validate the named answer before preparing a PIM update.
    matches = [a for a in decision.answers if a.name == "base_color"]
    if len(matches) != 1:
        return {"draft": None, "review": "Missing or duplicate base_color answer"}
    answer = matches[0]
    if answer.type != "choice" or answer.choice not in valid_keys:
        return {"draft": None, "review": "Refused, unclear, or invalid CVL key"}
    return {"draft": {"base_color": answer.choice}, "review": None}

In [ ]:
single_cvl_decision = None
if RUN_DECISIONS:
    started = perf_counter()
    single_cvl_decision = require_client().decisions.create(
        model=DECISION_MODEL,
        input=EVIDENCE,
        questions=[{
            "type": "choice",
            "name": "base_color",
            "instructions": "Map navy to blue. If unclear use needs_review.",
            "choices": [{"value": v} for v in ["blue", "red", "green", "needs_review"]],
        }],
    )
    show_decision(single_cvl_decision, started)
    print("PIM proposal:", color_draft(single_cvl_decision, {"blue", "red", "green"}))
else:
    print("Skipped: set RUN_DECISIONS = True for live Decisions examples.")

## 6. Decisions with image evidence — slide 11
The bundled photo depicts an **orange jacket**. This is a separate product example from the navy jacket text. Preview it, then send it as a base64 image content part. The base64 string is never printed.

The model selects a visible color; the image does not establish suitable activities, waterproofing, or certification. Compare the returned key with what you can see. Images can also be supplied through publicly accessible HTTP/HTTPS URLs.

Source: [Decisions image inputs](https://developers.openai.com/api/docs/guides/decisions).

In [ ]:
if not IMAGE_PATH.is_file():
    raise FileNotFoundError(f"Missing slide evidence asset: {IMAGE_PATH}")
display(Image(filename=str(IMAGE_PATH), width=360))

In [ ]:
image_decision = None
if RUN_DECISIONS:
    image_url = "data:image/png;base64," + b64encode(IMAGE_PATH.read_bytes()).decode("ascii")
    colors = ["orange", "blue", "black", "needs_review"]
    started = perf_counter()
    image_decision = require_client().decisions.create(
        model=DECISION_MODEL,
        input=[{
            "role": "user",
            "content": [{"type": "input_image", "image_url": image_url}],
        }],
        questions=[{
            "type": "choice",
            "name": "base_color",
            "instructions": "Select jacket color. If unclear use needs_review.",
            "choices": [{"value": color} for color in colors],
        }],
    )
    show_decision(image_decision, started)
    print("PIM proposal:", color_draft(image_decision, {"orange", "blue", "black"}))
else:
    print("Skipped: image preview is local; the API request needs RUN_DECISIONS = True.")

## 7. Multi-CVL: evaluate each candidate — slide 12
A multi-value field needs multiple independent judgments. Ask one **predicate per CVL key in the same request**, then collect supported keys.

Our evidence explicitly lists hiking and cycling; it does not list running. The question is about what is stated, rather than guessing all possible jacket uses. The thresholds below are illustrative: ≥0.85 selects; ≤0.15 excludes; the middle goes to review. Calibrate them on labeled examples before production. A predicate probability is a model estimate, not a proven guarantee.

A refusal, unexpected answer type, or missing answer also goes to review. The final application must validate CVL keys before writing changes.

In [ ]:
ACTIVITY_KEYS = ["hiking", "running", "cycling"]
SELECT_THRESHOLD, REJECT_THRESHOLD = 0.85, 0.15

def activity_draft(decision):
    selected, review = [], []
    for key in ACTIVITY_KEYS:
        matches = [a for a in decision.answers if a.name == key]
        if len(matches) != 1 or matches[0].type != "predicate":
            review.append(key)
            continue
        probability = matches[0].probability
        if not 0 <= probability <= 1:
            review.append(key)
        elif probability >= SELECT_THRESHOLD:
            selected.append(key)
        elif probability > REJECT_THRESHOLD:
            review.append(key)
    unknown = [a.name for a in decision.answers if a.name not in ACTIVITY_KEYS]
    return {
        "activities": selected,
        "review": review,
        "unexpected_answers": unknown,
        "ready_for_reviewed_write": not review and not unknown,
    }

multi_cvl_decision = None
if RUN_DECISIONS:
    started = perf_counter()
    multi_cvl_decision = require_client().decisions.create(
        model=DECISION_MODEL,
        input=EVIDENCE,
        questions=[{
            "type": "predicate",
            "name": key,
            "instructions": f"Is {key} explicitly listed as a suitable activity?",
        } for key in ACTIVITY_KEYS],
    )
    show_decision(multi_cvl_decision, started)
    print("PIM proposal:", activity_draft(multi_cvl_decision))
else:
    print("Skipped: set RUN_DECISIONS = True to evaluate the multi-CVL field.")

## 8. Predicate and an easy score — slide 16
For “Easy to use. I love it.”, ask whether the review is positive, then score its tone.

| Level index | Label |
|---|---|
| 0 | Negative |
| 1 | Neutral |
| 2 | Positive |

The score is the probability-weighted average of these **zero-based level indices**. A hypothetical 1.8 is near the positive end of this 0–2 rubric; it is neither a star rating nor 90% confidence. The cell prints the actual result and, when present, its level distribution.

In [ ]:
sentiment_decision = None
if RUN_DECISIONS:
    started = perf_counter()
    sentiment_decision = require_client().decisions.create(
        model=DECISION_MODEL,
        input="Review: Easy to use. I love it.",
        questions=[
            {"type": "predicate", "name": "positive",
             "instructions": "Is the review positive?"},
            {"type": "score", "name": "sentiment",
             "instructions": "Rate the tone of this review.",
             "levels": [
                 {"label": "Negative", "description": "An unfavorable opinion."},
                 {"label": "Neutral", "description": "Neither positive nor negative."},
                 {"label": "Positive", "description": "A favorable opinion."},
             ]},
        ],
    )
    show_decision(sentiment_decision, started)
    for answer in sentiment_decision.answers:
        if answer.type == "score":
            weighted_score = sum(p.value * p.probability for p in answer.probabilities)
            print(f"Sentiment: {answer.score:.3f} on a 0–2 scale")
            print(f"From returned level probabilities: {weighted_score:.3f}")
        elif answer.type == "refusal":
            print("Review needed for:", answer.name)
else:
    print("Skipped: set RUN_DECISIONS = True for predicate and score.")

## 9. Optional: separate Agents SDK — slides 4 and 17
The managed API above uses the `openai` package. This alternative uses **`openai-agents`**, where your Python application orchestrates the agent.

The project requirements now include this separate package. If your current kernel still reports a missing `agents` module, install the requirements into that kernel, restart it, rerun setup, and enable `RUN_AGENTS_SDK`:

```python
%pip install -r requirements.txt
```

`Agent()` creates a local configuration object. It does not create a managed saved `agent_id`. Reuse the code/configuration in other processes and choose the SDK's own session/history persistence. This text-only SDK example does not provision an OpenAI-hosted sandbox. In Jupyter, use top-level `await`.

Source: [Agents SDK quickstart](https://developers.openai.com/api/docs/guides/agents/quickstart).

In [ ]:
if RUN_AGENTS_SDK:
    require_client()
    try:
        from agents import Agent, Runner
    except ModuleNotFoundError as exc:
        if exc.name != "agents":
            raise
        raise RuntimeError(
            "The separate OpenAI Agents SDK is missing from this kernel. "
            "Run %pip install -r requirements.txt in a notebook cell, "
            "then restart the kernel, rerun setup, and retry this section."
        ) from None
    sdk_agent = Agent(
        name="Technical explainer",
        model=AGENT_MODEL,
        instructions="Explain technical issues clearly.",
    )
    sdk_result = await Runner.run(sdk_agent, "Why might a file export fail?")
    print(sdk_result.final_output)
else:
    print("Skipped: install openai-agents and set RUN_AGENTS_SDK = True.")

## 10. Optional: Decisions on Azure — slide 18
This requires a separate Azure OpenAI resource, key, and supported `gpt-6-luna` deployment. Add these environment variables to your root `.env`, then rerun setup:

```dotenv
AZURE_OPENAI_ENDPOINT=https://YOUR-RESOURCE.openai.azure.com
AZURE_OPENAI_API_KEY=YOUR_AZURE_KEY
AZURE_OPENAI_DEPLOYMENT=YOUR_DEPLOYMENT_NAME
```

The endpoint variable is the **resource root**; the code appends `/openai/v1/`. The model argument is your deployment name. Microsoft documents regional rollout restrictions, including East US 2 being excluded as of the documentation check. Verify availability for your resource. This section demonstrates Azure Decisions, independently of the managed Agents examples.

Source: [Microsoft: Azure OpenAI Decisions](https://learn.microsoft.com/en-us/azure/foundry/openai/how-to/decisions).

In [ ]:
if RUN_AZURE_DECISIONS:
    azure_names = ["AZURE_OPENAI_ENDPOINT", "AZURE_OPENAI_API_KEY", "AZURE_OPENAI_DEPLOYMENT"]
    missing = [name for name in azure_names if not os.getenv(name)]
    if missing:
        print("Azure demo skipped. Set:", ", ".join(missing))
    else:
        started = perf_counter()
        with OpenAI(
            base_url=os.environ["AZURE_OPENAI_ENDPOINT"].rstrip("/") + "/openai/v1/",
            api_key=os.environ["AZURE_OPENAI_API_KEY"],
            timeout=180,
            max_retries=0,
        ) as azure:
            azure_decision = azure.decisions.create(
                model=os.environ["AZURE_OPENAI_DEPLOYMENT"],
                input=EVIDENCE,
                questions=[{
                    "type": "predicate", "name": "hiking",
                    "instructions": "Is hiking explicitly listed?",
                }],
            )
        show_decision(azure_decision, started)
else:
    print("Skipped: Azure needs separate settings and RUN_AZURE_DECISIONS = True.")

## 11. Hosted costs and cleanup — slides 19 and 20
Managed-agent cost comprises **model tokens + hosted container usage + any separately priced tools**. The inline example requests an OpenAI-hosted sandbox; the saved-agent `none` example does not.

| Agents sandbox size | Memory | Standard USD per 20-minute container session |
|---|---|---|
| small | 1 GB | $0.03 |
| medium (default used here) | 4 GB | $0.12 |
| large | 16 GB | $0.48 |

Eligible container sessions can use per-minute billing with a five-minute minimum; check which billing applies to your account/session. These figures describe container pricing, not a guaranteed total for a notebook run. Check current [OpenAI pricing](https://developers.openai.com/api/docs/pricing#tools) and [hosted sandbox configuration](https://developers.openai.com/api/docs/guides/agents-api/environments/openai-hosted).

**Cleanup is explicit.** Copy a demo session ID into the list below only after saving needed artifacts. Deletion removes remote session state; closing Jupyter or the stream is not cleanup. Delete the saved agent configuration separately if you no longer want to reuse it. Default lists are empty.

In [ ]:
# First inspect state in memory. No API requests occur here.
print("Known demo IDs:", state)

# Paste only resources you intend to remove.
DELETE_SESSION_IDS = []  # Example: [state["inline_session_id"]]
DELETE_AGENT_IDS = []    # Example: [state["saved_agent_id"]]

for session_id in DELETE_SESSION_IDS:
    deletion = require_client().beta.agents.sessions.delete(session_id)
    print("Deleted session:", session_id)
    state = {k: v for k, v in state.items() if v != session_id}

for agent_id in DELETE_AGENT_IDS:
    deletion = require_client().beta.agents.delete(agent_id)
    print("Deleted saved agent:", agent_id)
    state = {k: v for k, v in state.items() if v != agent_id}

if DELETE_SESSION_IDS or DELETE_AGENT_IDS:
    STATE_PATH.write_text(json.dumps(state, indent=2), encoding="utf-8")
else:
    print("No cleanup requested.")

## 12. Try it yourself — slides 13–15 and 21
1. Change the supplier color to “forest green”. Does the single-CVL draft use `green`?
2. Remove color evidence entirely. Does the request select `needs_review`?
3. Add “running” to the stated activities. Compare the predicates before and after.
4. Replace the review with “It works, but setup was difficult.” Inspect the sentiment distribution.
5. Restart the kernel and retrieve a saved agent/session using only the setup and retrieval sections.

**Common issues:** a missing client method usually means the kernel uses an older SDK; check the version printed in setup. For 401/403 check key/project permissions and feature access. A 404 on retrieval can mean the ID belongs to another project or was deleted. For Azure, check deployment name and regional support. Do not blindly repeat session creation after a timeout: use any persisted session ID to inspect whether work already started.

**Production direction:** version configurations in source control, persist resource IDs with application conversation state, handle refusals and failed turns, validate draft changes, evaluate thresholds, and monitor usage. Use ordinary rules for exact mappings, Decisions for bounded judgments, and agents for tasks needing tools and multiple steps.

Further reading: [Agents API overview](https://developers.openai.com/api/docs/guides/agents-api/overview), [configuration](https://developers.openai.com/api/docs/guides/agents-api/configuration), [sessions](https://developers.openai.com/api/docs/guides/agents-api/sessions), [Decisions reference](https://developers.openai.com/api/reference/resources/decisions/methods/create), and [Agents SDK quickstart](https://developers.openai.com/api/docs/guides/agents/quickstart).